In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from timm import create_model

class ResViT(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # ResNet50
        self.cnn = models.resnet50(weights="IMAGENET1K_V1")
        cnn_dim = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        # ViT
        self.vit = create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        vit_dim = 768

        # Classifier (paper MLP)
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(cnn_dim + vit_dim),
            nn.Linear(cnn_dim + vit_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f_cnn = self.cnn(x)
        f_vit = self.vit(x)

        # CONCAT (critical)
        f = torch.cat([f_cnn, f_vit], dim=1)

        return self.classifier(f)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ResViT().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, epochs=20):
    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in tqdm(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()

        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.4f}")

    return model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return correct / total

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Use last conv layer of ResNet
target_layers = [model.cnn.layer4[-1]]

cam = GradCAM(model=model, target_layers=target_layers)

def apply_gradcam(image_tensor):
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0))

    cam_image = show_cam_on_image(
        image_tensor.permute(1,2,0).cpu().numpy(),
        grayscale_cam[0],
        use_rgb=True
    )

    return cam_image

In [ ]:
model = train_model(model, train_loader, val_loader, epochs=20)

test_acc = evaluate(model, test_loader)

print(f"Final Test Accuracy: {test_acc:.4f}")